# SRA Native LLM Integration PoC (TinyLlama)

このノートブックは、SRAのルーターを既存のLLM（ここではLlamaアーキテクチャの軽量版であるTinyLlama）の次元数に合わせてネイティブに統合する概念実証（PoC）です。
Google Colab (無料枠のT4 GPU等) で動作するように設計されています。

In [ ]:
# Google Colab等のための環境構築
!pip install -q transformers torch numpy

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM

# デバイスの設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# TinyLlamaのロード (Llamaアーキテクチャの軽量版, 約1.1Bパラメータ)
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_llm = AutoModelForCausalLM.from_pretrained(model_id).to(device)

# LLMの重みを凍結（SRAルーターのみを学習させるため）
for param in base_llm.parameters():
    param.requires_grad = False

# TinyLlamaの隠れ層の次元数を取得 (通常は2048次元)
hidden_size = base_llm.config.hidden_size
print(f"LLM Hidden Size (SRA Native Dimension): {hidden_size}")

## SRAルーターと外部モジュール（シナプス）の定義

SRAのベースシステムを「LLMと同じ次元数」で構築します。これによりプロジェクション層が不要になります。

In [ ]:
class SRARouter(nn.Module):
    def __init__(self, d_model, num_experts=2):
        super().__init__()
        # 極めて軽量なルーティング層（LLM本体の学習に比べてコストはほぼゼロ）
        self.gate = nn.Linear(d_model, num_experts)
        
    def forward(self, x):
        # x shape: [batch, seq_len, d_model]
        routing_logits = self.gate(x)
        routing_probs = torch.softmax(routing_logits, dim=-1)
        return routing_probs

class DummyCalculatorSynapse(nn.Module):
    """
    非学習の外部モジュール（計算機など）のモック。
    ネイティブ統合のため、LLMと同じd_modelを受け取り、d_modelを返す。
    """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # 実際にはここに計算機へテキストを渡し、結果をベクトル化する処理が入る
        
    def forward(self, x):
        # ここではモックとして、単にテンソルを0.5倍して返す（何らかの処理の代わり）
        print("  [Calculator Synapse] Activated for calculation!")
        return x * 0.5

## ネイティブ統合フォワードパスの検証

入力テキストをLLMのトークナイザーでベクトル化し、ルーターで「LLM本体」と「計算機」に振り分けるシミュレーションを行います。

In [ ]:
# 1. SRAシステムの初期化
router = SRARouter(d_model=hidden_size, num_experts=2).to(device)
calculator_synapse = DummyCalculatorSynapse(d_model=hidden_size).to(device)

# 2. 入力データの準備
text = "What is the result of 15 * 24 ?"
inputs = tokenizer(text, return_tensors="pt").to(device)

print(f"Input Text: '{text}'")
print(f"Token IDs shape: {inputs.input_ids.shape}")

# 3. LLMのEmbedding層でネイティブなベクトルに変換
with torch.no_grad():
    # TinyLlamaのEmbedding層を通す
    embeddings = base_llm.model.embed_tokens(inputs.input_ids)
print(f"Embeddings shape (Native): {embeddings.shape}")

# 4. ルーティングの実行
# プロジェクション層なしで直接ルーターに渡せる！
routing_probs = router(embeddings)
print(f"Routing Probabilities shape: {routing_probs.shape}")

# 5. ルーティング結果に基づく処理の分岐（シミュレーション）
# トークンごとに、LLM(Expert 0)に送るか、Calculator(Expert 1)に送るかを決める
# ここでは簡略化のため、最後のトークン（'?'）のルーティング確率を見る
last_token_probs = routing_probs[0, -1, :]
print(f"Routing Probs for last token: LLM={last_token_probs[0]:.4f}, Calc={last_token_probs[1]:.4f}")

# 実際のSRAでは、ここで確率に応じて処理を分岐・結合します
print("\n--- Routing Execution ---")
if last_token_probs[1] > 0.5:
    # 計算機にルーティングされた場合（プロジェクションなしで直接渡せる！）
    out_tensor = calculator_synapse(embeddings)
else:
    # LLMにルーティングされた場合
    print("  [LLM Synapse] Processing via Frozen LLM Layers...")
    # 本来はTransformerブロックを通す
    out_tensor = embeddings 

print(f"Output Tensor shape: {out_tensor.shape}")
print("\n✅ ネイティブ統合（次元変換なしのシームレスな接続）の検証完了！")